In [6]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
np.set_printoptions(threshold=sys.maxsize)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
    board_w=10,
    board_h=20,
    vanish_zone=4, # Extra rows above the visible board to capture piece spawns
)

CONFIG = Configuration(
    max_placements=50,
    max_board_size_w=10,
    max_board_size_h=20,
)

# Train

In [24]:
from src.tetris import Board, PieceEnum, Queue, ActionEnum, ActivePiece, Tetris, RotationEnum, ROTATION_DIR

In [26]:
from src.models import TetrisEnv

env = TetrisEnv(CONFIG, T_CONFIG)

In [30]:
print(env.reset()[0]["boards"].shape)
print(env.reset()[0]["queue"].shape)

Found 17 valid placements for piece I
(50, 24, 10)
Found 17 valid placements for piece I
(7, 8)


In [21]:
boards = env.reset()[0]["boards"]

Found 10 valid placements for piece L


In [22]:
boards

array([[[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 1.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1.

### Model

In [11]:
env.observation_space["boards"].shape

(50, 14, 30)

In [12]:
from src.models import TurboMino

feature_extractor = TurboMino(
    T_CONFIG,
    CONFIG,
    env.observation_space,
)

### Correct forward pass (current model)

The feature extractor returns a single tensor `(B, max_placements)` — one scalar per placement.
The env provides a `placement_mask` so the model masks out invalid/padded slots with `-1e9`.

In [13]:
import torch

obs, _ = env.reset()
tensor_obs = {
    "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
    "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
    "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
}
print("boards:", tensor_obs["boards"][0].shape)          # (10, 4)
print("queue:", tensor_obs["queue"][0].shape)           # (5, 4)

values = feature_extractor(tensor_obs)
print("Output shape:", values.shape)                # (1, 50)
print("Mask:", tensor_obs["placement_mask"][0])
print("Valid placements:", tensor_obs["placement_mask"].sum().item())
print("Placement values:\n", values)
print("Best placement:", values.argmax().item())

Found 5 valid placements for piece S
boards: torch.Size([50, 14, 30])
queue: torch.Size([7, 8])
Output shape: torch.Size([1, 50])
Mask: tensor([ True,  True,  True,  True,  True, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False,
        False, False, False, False, False, False, False, False, False, False])
Valid placements: 5
Placement values:
 tensor([[-4.1139e-02, -4.1490e-02, -4.1722e-02, -4.1349e-02, -4.1411e-02,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09,
         -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.0000e+09, -1.

In [14]:
# import time 
# import tqdm

# total_iters = 4*5

# print('warming up...')
# for i in tqdm.tqdm(range(total_iters*1000)):
#     obs, _ = env.reset()
#     tensor_obs = {
#         "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
#         "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
#         "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
#     }

#     values = feature_extractor(tensor_obs)


# t1 = time.time()
# for i in range(total_iters):
#     obs, _ = env.reset()
#     tensor_obs = {
#         "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
#         "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
#         "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
#     }

#     values = feature_extractor(tensor_obs)

# t2 = time.time()

# print(f"Average inference time over {total_iters} iterations: {(t2 - t1) / total_iters:.4f} seconds")